In [ ]:
# code by Edward Saunders for David Reber

In [1]:
import datetime
import os
from galvani import BioLogic
import json
import yadg

In [2]:
def datetime_to_unix_seconds(dt: datetime.datetime) -> float:
    """Convert a datetime object to Unix epoch time.

    Args:
        dt (datetime.datetime): The datetime object to convert.

    Returns:
        float: The Unix epoch time in seconds.
    """
    unix_epoch = datetime.datetime(1970, 1, 1) # Convert to seconds since Unix epoch
    seconds = (dt - unix_epoch).total_seconds()
    return seconds

def convert_fpath(fpath: str) -> str:
    """Convert the file path to an absolute path and extract the file type.

    Args:
        fpath (str): The file path to convert.

    Returns:
        str: The absolute file path.
    """
    return os.path.abspath(fpath)

def extract_mpr_galvani(fpath: str) -> BioLogic.MPRfile:
    """Extract the MPR file from the given file path.

    Args:
        fpath (str): The file path to the MPR file.

    Returns:
        BioLogic.MPRfile: The extracted MPR file.
    """
    fpath = convert_fpath(fpath)
    data_file = BioLogic.MPRfile(fpath)
    return data_file, data_file.timestamp

def ole_to_datetime(ole_timestamp):
    # OLE base date is 1899-12-30
    ole_base = datetime.datetime(1899, 12, 30)
    dt = ole_base + datetime.timedelta(days=ole_timestamp)
    return dt

def extract_mpr_yadg(fpath: str):
    datatree = yadg.extractors.extract(filetype="eclab.mpr", path=fpath)
    ole_timestamp = json.loads(datatree.attrs["original_metadata"])["log"]["ole_timestamp"]
    dt = ole_to_datetime(ole_timestamp)
    return datatree, dt


In [ ]:
### Your file here
fpath = r"C:\Users\es758\Downloads\P29_cell1_EIS-polarization-CA_fastflow_01_PEIS_C01.mpr"

try:
    data_file, timestamp = extract_mpr_galvani(fpath)
    startdate = data_file.startdate
    enddate = data_file.enddate
    start_timestamp = data_file.timestamp
    print(f"Start date: {startdate}")
    print(f"End date: {enddate}")
    print(f"Start timestamp: {start_timestamp}")
except Exception as e:
    print(f"Galvani failed ({e}), using yadg fallback")
    datatree, timestamp = extract_mpr_yadg(fpath)
    print(f"Timestamp: {timestamp}")

Start date: 2025-08-15
End date: 2025-08-15
Start timestamp: 2025-08-15 12:45:13.885000
Data file dtype: [('freq/Hz', '<f4'), ('Re(Z)/Ohm', '<f4'), ('-Im(Z)/Ohm', '<f4'), ('|Z|/Ohm', '<f4'), ('Phase(Z)/deg', '<f4'), ('time/s', '<f8'), ('<Ewe>/V', '<f4'), ('<I>/mA', '<f4'), ('Cs/µF', '<f4'), ('Cp/µF', '<f4'), ('cycle number', '<f8'), ('I Range', '<u2'), ('|Ewe|/V', '<f4'), ('|I|/A', '<f4'), ('Ns', '<u2'), ('(Q-Qo)/mA.h', '<f8')]
